## Challenge A

In [20]:
import numpy as np
from matplotlib import pyplot as plt
import numpy as np
from ngsolve import *
from netgen.geom2d import SplineGeometry

# Finite Difference

In [19]:
def build_up_b(b, rho, dt, u, v, dx, dy):
    """Calculates the source term for the Pressure Poisson Equation."""
    b[1:-1, 1:-1] = (rho * (1 / dt * ((u[1:-1, 2:] - u[1:-1, 0:-2]) / (2 * dx) + 
                     (v[2:, 1:-1] - v[0:-2, 1:-1]) / (2 * dy)) -
                    ((u[1:-1, 2:] - u[1:-1, 0:-2]) / (2 * dx))**2 -
                    2 * ((u[2:, 1:-1] - u[0:-2, 1:-1]) / (2 * dy) *
                         (v[1:-1, 2:] - v[1:-1, 0:-2]) / (2 * dx)) -
                    ((v[2:, 1:-1] - v[0:-2, 1:-1]) / (2 * dy))**2))
    return b

def pressure_poisson(p, dx, dy, b, nit):
    """Iteratively solves the Poisson equation to find the pressure field."""
    pn = np.empty_like(p)
    for q in range(nit):
        pn = p.copy()
        p[1:-1, 1:-1] = (((pn[1:-1, 2:] + pn[1:-1, 0:-2]) * dy**2 + 
                          (pn[2:, 1:-1] + pn[0:-2, 1:-1]) * dx**2) /
                          (2 * (dx**2 + dy**2)) -
                          dx**2 * dy**2 / (2 * (dx**2 + dy**2)) * b[1:-1, 1:-1])

        # Boundary Conditions for Pressure
        p[:, -1] = 0        # outflow
        p[:, 0] = p[:, 1]   # inflow
        p[0, :] = p[1, :]   # bottom
        p[-1, :] = p[-2, :] # top
    return p

def vortex_street_fd(nt, nx, ny, dt, dx, dy, rho, nu, nit):
    u = np.zeros((ny, nx))
    v = np.zeros((ny, nx))
    p = np.zeros((ny, nx))
    b = np.zeros((ny, nx))
    
    # Define the cylinder obstacle (mask)
    cx, cy, r = 0.5, 1.0, 0.1
    x = np.linspace(0, 2, nx)
    y = np.linspace(0, 2, ny)
    X, Y = np.meshgrid(x, y)
    cylinder_mask = (X - cx)**2 + (Y - cy)**2 < r**2

    for n in range(nt):
        un, vn = u.copy(), v.copy()
        b = build_up_b(b, rho, dt, u, v, dx, dy)
        p = pressure_poisson(p, dx, dy, b, nit)

        # Update Velocity
        u[1:-1, 1:-1] = (un[1:-1, 1:-1] -
                         un[1:-1, 1:-1] * dt / dx * (un[1:-1, 1:-1] - un[1:-1, 0:-2]) -
                         vn[1:-1, 1:-1] * dt / dy * (un[1:-1, 1:-1] - un[0:-2, 1:-1]) -
                         dt / (2 * rho * dx) * (p[1:-1, 2:] - p[1:-1, 0:-2]) +
                         nu * (dt / dx**2 * (un[1:-1, 2:] - 2 * un[1:-1, 1:-1] + un[1:-1, 0:-2]) +
                               dt / dy**2 * (un[2:, 1:-1] - 2 * un[1:-1, 1:-1] + un[0:-2, 1:-1])))

        v[1:-1, 1:-1] = (vn[1:-1, 1:-1] -
                         un[1:-1, 1:-1] * dt / dx * (vn[1:-1, 1:-1] - vn[1:-1, 0:-2]) -
                         vn[1:-1, 1:-1] * dt / dy * (vn[1:-1, 1:-1] - vn[0:-2, 1:-1]) -
                         dt / (2 * rho * dy) * (p[2:, 1:-1] - p[0:-2, 1:-1]) +
                         nu * (dt / dx**2 * (vn[1:-1, 2:] - 2 * vn[1:-1, 1:-1] + vn[1:-1, 0:-2]) +
                               dt / dy**2 * (vn[2:, 1:-1] - 2 * vn[1:-1, 1:-1] + vn[0:-2, 1:-1])))

        # Apply No-Slip Boundary Condition
        u[cylinder_mask] = 0
        v[cylinder_mask] = 0

        # External Boundary Conditions
        u[:, 0] = 1.0
        u[:, -1] = u[:, -2]
        v[:, 0] = 0
        v[:, -1] = v[:, -2]
        u[0, :], u[-1, :], v[0, :], v[-1, :] = 0, 0, 0, 0 # Walls
        
    return u, v, p

# Parameters
nx, ny, nt, nit = 101, 51, 500, 50
dx, dy = 2 / (nx - 1), 2 / (ny - 1)
dt, rho, nu = 0.001, 1.0, 0.01 

u, v, p = vortex_street_fd(nt, nx, ny, dt, dx, dy, rho, nu, nit)

def run_stability_test(start_nu, decrease_factor, u_in, d):
    current_nu = start_nu
    stable = True
    
    print(f"{'nu':<10} | {'Re':<10} | {'Status'}")
    print("-" * 35)
    
    while stable:
        # Calculate current Re
        Re = (u_in * d) / current_nu
        
        # Run simulation
        try:
            u, v, p = vortex_street_fd(nt=300, nx=101, ny=51, dt=0.001, 
                                       dx=0.02, dy=0.04, rho=1.0, 
                                       nu=current_nu, nit=50)
            
            # CHECK 1: Numerical divergence
            if np.isnan(u).any() or np.isnan(p).any():
                print(f"{current_nu:.5f} | {Re:<10.2f} | BROKEN (Numerical Divergence)")
                stable = False
            
            # CHECK 2: Look for unrealistic velocities
            elif np.max(np.abs(u)) > 10.0:
                print(f"{current_nu:.5f} | {Re:<10.2f} | BROKEN (Velocity Divergence)")
                stable = False
            
            else:
                print(f"{current_nu:.5f} | {Re:<10.2f} | Stable")
                # Decrease nu for the next round
                current_nu *= decrease_factor
                
        except Exception as e:
            print(f"{current_nu:.5f} | {Re:<10.2f} | FAILED ({e})")
            stable = False

    return current_nu

# Parameters for the test
inlet_velocity = 1.0
cylinder_diameter = 0.2
starting_nu = 0.01
factor = 0.7 # Decrease nu by 30% each step

breaking_point_nu = run_stability_test(starting_nu, factor, inlet_velocity, cylinder_diameter)

nu         | Re         | Status
-----------------------------------
0.01000 | 20.00      | Stable
0.00700 | 28.57      | Stable
0.00490 | 40.82      | Stable
0.00343 | 58.31      | Stable
0.00240 | 83.30      | BROKEN (Numerical Divergence)


C:\Users\sofro\AppData\Local\Temp\ipykernel_7100\57054851.py:5: RuntimeWarning: overflow encountered in square
  ((u[1:-1, 2:] - u[1:-1, 0:-2]) / (2 * dx))**2 -
C:\Users\sofro\AppData\Local\Temp\ipykernel_7100\57054851.py:6: RuntimeWarning: overflow encountered in multiply
  2 * ((u[2:, 1:-1] - u[0:-2, 1:-1]) / (2 * dy) *
C:\Users\sofro\AppData\Local\Temp\ipykernel_7100\57054851.py:3: RuntimeWarning: invalid value encountered in subtract
  b[1:-1, 1:-1] = (rho * (1 / dt * ((u[1:-1, 2:] - u[1:-1, 0:-2]) / (2 * dx) +
C:\Users\sofro\AppData\Local\Temp\ipykernel_7100\57054851.py:8: RuntimeWarning: overflow encountered in square
  ((v[2:, 1:-1] - v[0:-2, 1:-1]) / (2 * dy))**2))
C:\Users\sofro\AppData\Local\Temp\ipykernel_7100\57054851.py:48: RuntimeWarning: overflow encountered in multiply
  un[1:-1, 1:-1] * dt / dx * (un[1:-1, 1:-1] - un[1:-1, 0:-2]) -
C:\Users\sofro\AppData\Local\Temp\ipykernel_7100\57054851.py:49: RuntimeWarning: overflow encountered in multiply
  vn[1:-1, 1:-1] * dt / d

# Finite Element Method

In [21]:
def run_fem_stability_test(start_nu, factor):
    current_nu = start_nu
    stable = True
    
    print(f"{'nu':<10} | {'Re':<10} | {'Status'}")
    print("-" * 45)
    
    while stable:
        # Mesh
        geo = SplineGeometry()
        geo.AddRectangle(p1=(0, 0), p2=(2, 0.4), bcs=["wall", "outlet", "wall", "inlet"])
        geo.AddCircle(c=(0.4, 0.2), r=0.05, leftdomain=0, rightdomain=1, bc="cylinder", maxh=0.01)
        mesh = Mesh(geo.GenerateMesh(maxh=0.05))
        
        # Spaces
        V = VectorH1(mesh, order=2, dirichlet="wall|cylinder|inlet")
        Q = H1(mesh, order=1)
        X = V * Q
        gfu = GridFunction(X)
        u_in = 1.0
        gfu.components[0].Set(CoefficientFunction((u_in, 0)), definedon=mesh.Boundaries("inlet"))
        
        Re = (u_in * 0.1) / current_nu
        dt = 0.001
        (u, p), (v, q) = X.TrialFunction(), X.TestFunction()
        
        try:
            # Matrices
            m_rhs = BilinearForm(X); m_rhs += InnerProduct(u, v) * dx; m_rhs.Assemble()
            
            a_lhs = BilinearForm(X)
            a_lhs += InnerProduct(u, v) * dx 
            a_lhs += dt * (current_nu * InnerProduct(grad(u), grad(v)) + div(u)*q + div(v)*p) * dx
            a_lhs.Assemble()
            
            inv = a_lhs.mat.Inverse(X.FreeDofs()) 

            # for loop for time-stepping
            for step in range(300):
                conv = LinearForm(X)
                conv += -dt * InnerProduct(grad(gfu.components[0]) * gfu.components[0], v) * dx
                conv.Assemble()
                
                # Update
                res = m_rhs.mat * gfu.vec + conv.vec
                gfu.vec.data = inv * res
                
                # Stopping Criteria:
                
                # A. Velocity Magnitude Check
                u_vel = gfu.components[0]
                speed_max = max([abs(x) for x in gfu.components[0].vec])
                
                # B. Numerical Divergence (CFL)
                if speed_max > 50.0:
                    raise ValueError(f"High Velocity / CFL Crash: {speed_max:.2f}")

                # C. Symmetry/Numerical divergence
                if np.isnan(Norm(gfu.vec)):
                    raise ValueError("Numerical divergence")

            print(f"{current_nu:.5f} | {Re:<10.2f} | Stable")
            current_nu *= factor 
            
            # Stop if nu is so small it's effectively zero 
            if current_nu < 1e-8:
                print("--- Nu reached physical limit of the mesh ---")
                stable = False
            
        except Exception as e:
            print(f"{current_nu:.5f} | {Re:<10.2f} | BROKEN ({str(e)})")
            stable = False

run_fem_stability_test(start_nu=0.01, factor=0.7)

nu         | Re         | Status
---------------------------------------------
0.01000 | 10.00      | Stable
0.00700 | 14.29      | Stable
0.00490 | 20.41      | Stable
0.00343 | 29.15      | Stable
0.00240 | 41.65      | Stable
0.00168 | 59.50      | Stable
0.00118 | 85.00      | Stable
0.00082 | 121.43     | Stable
0.00058 | 173.47     | Stable
0.00040 | 247.81     | Stable
0.00028 | 354.01     | Stable
0.00020 | 505.73     | Stable
0.00014 | 722.48     | Stable
0.00010 | 1032.11    | Stable
0.00007 | 1474.44    | Stable
0.00005 | 2106.34    | Stable
0.00003 | 3009.06    | Stable
0.00002 | 4298.66    | Stable
0.00002 | 6140.95    | Stable
0.00001 | 8772.78    | Stable
0.00001 | 12532.54   | Stable
0.00001 | 17903.63   | Stable
0.00000 | 25576.62   | Stable
0.00000 | 36538.03   | Stable
0.00000 | 52197.18   | Stable
0.00000 | 74567.40   | Stable
0.00000 | 106524.86  | Stable
0.00000 | 152178.37  | Stable
0.00000 | 217397.67  | Stable
0.00000 | 310568.10  | Stable
0.00000 | 443668.71  

# LBM

In [22]:
# D2Q9 Constants
c = np.array([[0,0], [1,0], [0,1], [-1,0], [0,-1], [1,1], [-1,1], [-1,-1], [1,-1]])
w = np.array([4/9, 1/9, 1/9, 1/9, 1/9, 1/36, 1/36, 1/36, 1/36])
opp = np.array([0, 3, 4, 1, 2, 7, 8, 5, 6])

def equilibrium(rho, ux, uy):
    feq = np.zeros((rho.shape[0], rho.shape[1], 9))
    usqr = ux**2 + uy**2
    for i in range(9):
        cu = c[i, 0] * ux + c[i, 1] * uy
        feq[:, :, i] = w[i] * rho * (1.0 + 3.0*cu + 4.5*cu**2 - 1.5*usqr)
    return feq

def run_advanced_lbm_test(start_nu, factor):
    current_nu = start_nu
    # Start with a base resolution and increase it to keep tau stable
    base_r = 8 
    
    print(f"{'Resolution':<12} | {'nu':<8} | {'Re':<10} | {'tau':<8} | {'Status'}")
    print("-" * 60)
    
    for i in range(10):
        # Increase cylinder
        r_cyl = base_r + (i * 4) 
        Nx, Ny = r_cyl * 25, r_cyl * 10
        cx, cy = Nx // 5, Ny // 2
        
        D = 2 * r_cyl
        U_inlet = 0.1
        Re = (U_inlet * D) / current_nu
        tau = 3.0 * current_nu + 0.5
        
        # Setup Mask
        x, y = np.arange(Nx), np.arange(Ny)
        X, Y = np.meshgrid(x, y, indexing='ij')
        obstacle = (X - cx)**2 + (Y - cy)**2 <= r_cyl**2
        
        # Init
        rho = np.ones((Nx, Ny))
        ux = np.full((Nx, Ny), U_inlet)
        uy = 0.001 * U_inlet * np.sin(2.0 * np.pi * Y / Ny)
        ux[obstacle], uy[obstacle] = 0, 0
        f = equilibrium(rho, ux, uy)
        
        try:
            for step in range(600):
                rho = np.sum(f, axis=2)
                ux = np.sum(f * c[:, 0], axis=2) / rho
                uy = np.sum(f * c[:, 1], axis=2) / rho
                
                f_out = f - (f - equilibrium(rho, ux, uy)) / tau
                
                for j in range(9):
                    f_out[obstacle, j] = f[obstacle, opp[j]]
                    f[:, :, j] = np.roll(np.roll(f_out[:, :, j], c[j,0], axis=0), c[j,1], axis=1)
                
                # Outlet and Zou-He Inlet
                f[-1, :, :] = f[-2, :, :]
                rho_in = (f[0,:,0]+f[0,:,2]+f[0,:,4] + 2.*(f[0,:,3]+f[0,:,6]+f[0,:,7]))/(1.-U_inlet)
                f[0,:,1] = f[0,:,3] + (2./3.)*rho_in*U_inlet
                f[0,:,5] = f[0,:,7] - 0.5*(f[0,:,2]-f[0,:,4]) + (1./6.)*rho_in*U_inlet
                f[0,:,8] = f[0,:,6] + 0.5*(f[0,:,2]-f[0,:,4]) + (1./6.)*rho_in*U_inlet

                if np.isnan(np.min(f)) or np.max(np.abs(ux)) > 0.5:
                    raise ValueError("Diverged")

            print(f"D={D:<10} | {current_nu:.5f} | {Re:<10.2f} | {tau:.4f} | Stable")
            current_nu *= factor
            
        except:
            print(f"D={D:<10} | {current_nu:.5f} | {Re:<10.2f} | {tau:.4f} | BROKEN")
            break

run_advanced_lbm_test(0.01, 0.7)

Resolution   | nu       | Re         | tau      | Status
------------------------------------------------------------
D=16         | 0.01000 | 160.00     | 0.5300 | Stable
D=24         | 0.00700 | 342.86     | 0.5210 | BROKEN
